***Artificial Bee Colony Optimization***

Optimization algorithm inspired by the behavior of honey bees. The algorithm starts by creating a population of solutions, and iteratively improving them to find the optimal solution within three phases: 

    - Employed Bee: 

        Selects a solution and generates a new random solution based on it, if the new solution has a better fit, it is added to the population.

    - Onlooker Bee:

        Selects a solution based on the probability of the solution's fitness. Then does what the employed bee did.

    - Scout Bee: 
    
        This phase is activated if the candidate solution hasn't improved over a number of iterations, by generating a random solution based on the structure of the current solution.

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings("ignore")

X_df = pd.read_excel('minmax.xlsx')
y_df = pd.read_csv('idC_with_header.csv')

X = X_df.to_numpy()
y = y_df.to_numpy()

# Configuration
n_features = X.shape[1]
colony_size = 30
max_cycles = 100
limit = 20
classifier = GaussianNB()

# Fitness
def fitness(sol):
    if not sol.any():
        return 0
    X_selected = X[:, sol.astype(bool)]
    scores = cross_val_score(classifier, X_selected, y, cv=5, scoring='accuracy')
    return scores.mean()

# Initializing ABC structure
solutions = np.random.randint(0, 2, size=(colony_size, n_features))
fitness_vals = np.array([fitness(s) for s in solutions])
trial_counters = np.zeros(colony_size, dtype=int)

print("Started ABC")
# ABC loop
for cycle in range(max_cycles):
    print("Cycle: ", cycle)
    # Employed Bee
    for i in range(colony_size):
        k = np.random.choice([j for j in range(colony_size) if j != i])
        phi = np.random.uniform(-1, 1, size=n_features)
        candidate = solutions[i].copy()
        candidate[phi < 0] = solutions[k][phi < 0]
        fit_candidate = fitness(candidate)
        if fit_candidate > fitness_vals[i]:
            solutions[i], fitness_vals[i], trial_counters[i] = candidate, fit_candidate, 0
        else: 
            trial_counters[i] += 1
    
    # Onlooker Bee
    probs = fitness_vals / fitness_vals.sum()
    for _ in range(colony_size):
        i = np.random.choice(range(colony_size), p=probs)
        k = np.random.choice([j for j in range(colony_size) if j != i])
        phi = np.random.uniform(-1, 1, size=n_features)
        candidate = solutions[i].copy()
        candidate[phi < 0] = solutions[k][phi < 0]
        fit_candidate = fitness(candidate)
        if fit_candidate > fitness_vals[i]:
            solutions[i], fitness_vals[i], trial_counters[i] = candidate, fit_candidate, 0
        else:
            trial_counters[i] += 1
    
    # Scout Bee
    for i in range(colony_size):
        if trial_counters[i] >= limit:
            solutions[i]    = np.random.randint(0, 2, size=n_features)
            fitness_vals[i] = fitness(solutions[i])
            trial_counters[i] = 0

best_idx  = np.argmax(fitness_vals)
best_sol  = solutions[best_idx]
selected = np.where(best_sol == 1)[0]
print(f"Selected {len(selected)} features; Best CV Accuracy = {fitness_vals[best_idx]:.4f}")
print("Selected Features: ", selected)

# Running models:
X_best = X[:, best_sol.astype(bool)]
X_train, X_test, y_train, y_test = train_test_split(X_best, y, test_size=0.2, random_state=42, stratify=y)


svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_preds)
print(f"SVM Accuracy on features selected by ABC: {svm_acc * 100:.4f}")

nbc_model = GaussianNB()
nbc_model.fit(X_train, y_train)
nbc_preds = nbc_model.predict(X_test)
nbc_acc = accuracy_score(y_test, nbc_preds)
print(f"Naïve Bayes Accuracy on features selected by ABC: {nbc_acc * 100:.4f}")

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)
print(f"Random Forest Accuracy on features selected by ABC: {rf_acc * 100:.4f}")


Started ABC
Cycle:  0
Cycle:  1
Cycle:  2
Cycle:  3
Cycle:  4
Cycle:  5
Cycle:  6
Cycle:  7
Cycle:  8
Cycle:  9
Cycle:  10
Cycle:  11
Cycle:  12
Cycle:  13
Cycle:  14
Cycle:  15
Cycle:  16
Cycle:  17
Cycle:  18
Cycle:  19
Cycle:  20
Cycle:  21
Cycle:  22
Cycle:  23
Cycle:  24
Cycle:  25
Cycle:  26
Cycle:  27
Cycle:  28
Cycle:  29
Cycle:  30
Cycle:  31
Cycle:  32
Cycle:  33
Cycle:  34
Cycle:  35
Cycle:  36
Cycle:  37
Cycle:  38
Cycle:  39
Cycle:  40
Cycle:  41
Cycle:  42
Cycle:  43
Cycle:  44
Cycle:  45
Cycle:  46
Cycle:  47
Cycle:  48
Cycle:  49
Cycle:  50
Cycle:  51
Cycle:  52
Cycle:  53
Cycle:  54
Cycle:  55
Cycle:  56
Cycle:  57
Cycle:  58
Cycle:  59
Cycle:  60
Cycle:  61
Cycle:  62
Cycle:  63
Cycle:  64
Cycle:  65
Cycle:  66
Cycle:  67
Cycle:  68
Cycle:  69
Cycle:  70
Cycle:  71
Cycle:  72
Cycle:  73
Cycle:  74
Cycle:  75
Cycle:  76
Cycle:  77
Cycle:  78
Cycle:  79
Cycle:  80
Cycle:  81
Cycle:  82
Cycle:  83
Cycle:  84
Cycle:  85
Cycle:  86
Cycle:  87
Cycle:  88
Cycle:  89
Cycle:  

**AE and ABC**

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx') 
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model

class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        #Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh() 
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance losses
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32


# Initialize model, optimizer, and scheduler
model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train).numpy()
    test_latent = model.encoder(X_test).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

# Configuration
n_features = latent_dim
colony_size = 30
max_cycles = 100
limit = 20
classifier = GaussianNB()

# Fitness
def fitness(sol):
    if not sol.any():
        return 0
    X_selected = train_latent[:, sol.astype(bool)]
    scores = cross_val_score(classifier, X_selected, y_train_np, cv=5, scoring='accuracy')
    return scores.mean()

# Initializing ABC structure
solutions = np.random.randint(0, 2, size=(colony_size, n_features))
fitness_vals = np.array([fitness(s) for s in solutions])
trial_counters = np.zeros(colony_size, dtype=int)

print("Started ABC")
# ABC loop
for cycle in range(max_cycles):
    print("Cycle: ", cycle)
    # Employed Bee
    for i in range(colony_size):
        k = np.random.choice([j for j in range(colony_size) if j != i])
        phi = np.random.uniform(-1, 1, size=n_features)
        candidate = solutions[i].copy()
        candidate[phi < 0] = solutions[k][phi < 0]
        fit_candidate = fitness(candidate)
        if fit_candidate > fitness_vals[i]:
            solutions[i], fitness_vals[i], trial_counters[i] = candidate, fit_candidate, 0
        else: 
            trial_counters[i] += 1
    
    # Onlooker Bee
    probs = fitness_vals / fitness_vals.sum()
    for _ in range(colony_size):
        i = np.random.choice(range(colony_size), p=probs)
        k = np.random.choice([j for j in range(colony_size) if j != i])
        phi = np.random.uniform(-1, 1, size=n_features)
        candidate = solutions[i].copy()
        candidate[phi < 0] = solutions[k][phi < 0]
        fit_candidate = fitness(candidate)
        if fit_candidate > fitness_vals[i]:
            solutions[i], fitness_vals[i], trial_counters[i] = candidate, fit_candidate, 0
        else:
            trial_counters[i] += 1
    
    # Scout Bee
    for i in range(colony_size):
        if trial_counters[i] >= limit:
            solutions[i]    = np.random.randint(0, 2, size=n_features)
            fitness_vals[i] = fitness(solutions[i])
            trial_counters[i] = 0

best_idx  = np.argmax(fitness_vals)
best_sol  = solutions[best_idx]
selected = np.where(best_sol == 1)[0]
print(f"Selected {len(selected)} features; Best CV Accuracy = {fitness_vals[best_idx]:.4f}")
print("Selected Features: ", selected)

# Running models:
X_train_selected = train_latent[:, best_sol.astype(bool)]
X_test_selected = test_latent[:, best_sol.astype(bool)]

svm_model = SVC(random_state=42)
svm_model.fit(X_train_selected, y_train_np)
svm_preds = svm_model.predict(X_test_selected)
svm_acc = accuracy_score(y_test_np, svm_preds)
print(f"SVM Accuracy on features selected by ABC: {svm_acc * 100:.4f}")

nbc_model = GaussianNB()
nbc_model.fit(X_train_selected, y_train_np)
nbc_preds = nbc_model.predict(X_test_selected)
nbc_acc = accuracy_score(y_test_np, nbc_preds)
print(f"Naïve Bayes Accuracy on features selected by ABC: {nbc_acc * 100:.4f}")

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_selected, y_train_np)
rf_preds = rf_model.predict(X_test_selected)
rf_acc = accuracy_score(y_test_np, rf_preds)
print(f"Random Forest Accuracy on features selected by ABC: {rf_acc * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.2036
Epoch [2/50], Loss: 0.7624
Epoch [3/50], Loss: 0.6490
Epoch [4/50], Loss: 0.5489
Epoch [5/50], Loss: 0.4880
Epoch [6/50], Loss: 0.4904
Epoch [7/50], Loss: 0.5256
Epoch [8/50], Loss: 0.4252
Epoch [9/50], Loss: 0.4432
Epoch [10/50], Loss: 0.4251
Epoch [11/50], Loss: 0.4879
Epoch [12/50], Loss: 0.4632
Epoch [13/50], Loss: 0.4235
Epoch [14/50], Loss: 0.3552
Epoch [15/50], Loss: 0.3719
Epoch [16/50], Loss: 0.4566
Epoch [17/50], Loss: 0.3952
Epoch [18/50], Loss: 0.4149
Epoch [19/50], Loss: 0.3981
Epoch [20/50], Loss: 0.4255
Epoch [21/50], Loss: 0.3539
Epoch [22/50], Loss: 0.3716
Epoch [23/50], Loss: 0.4128
Epoch [24/50], Loss: 0.4030
Epoch [25/50], Loss: 0.4449
Epoch [26/50], Loss: 0.3872
Epoch [27/50], Loss: 0.3906
Epoch [28/50], Loss: 0.3590
Epoch [29/50], Loss: 0.4249
Epoch [30/50], Loss: 0.3937
Epoch [31/50], Loss: 0.3849
Epoch [32/50], Loss: 0.3749
Epoch [33/50], Loss: 0.4240
Epoch [34/50], Loss: 0.3667
Epoch [35/50], Loss: 0.3914


**VAE + ABC**

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx')
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Variational Autoencoder-Classifier Model

class JointVAEClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointVAEClassifier, self).__init__()
        # Encoder
        self.encoder_net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128)
        )
        # Layers to produce the mean and log-variance for the latent distribution
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        # Encode input to get intermediate representation
        x_encoded = self.encoder_net(x)
        # Produce latent distribution parameters
        mu = self.fc_mu(x_encoded)
        logvar = self.fc_logvar(x_encoded)
        # Sample latent vector
        z = self.reparameterize(mu, logvar)
        # Reconstruct input and classify based on latent vector
        reconstruction = self.decoder(z)
        logits = self.classifier(z)
        return reconstruction, logits, mu, logvar


# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance reconstruction and KL loss and classification loss
lambda_recon = 0.5

def vae_combined_loss(reconstructed, original, logits, labels, mu, logvar):
    # Reconstruction loss
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    # KL divergence loss: average over batch
    loss_kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    # Classification loss
    loss_class = classification_loss_fn(logits, labels)
    # Combined loss: balance reconstruction (with KL) and classification
    return lambda_recon * (loss_recon + loss_kl) + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

# Initialize model, optimizer, and scheduler
model = JointVAEClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50


# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits, mu, logvar = model(inputs)
        loss = vae_combined_loss(reconstruction, inputs, logits, labels, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")


# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_encoded = model.encoder_net(X_train)
    train_latent = model.fc_mu(train_encoded).numpy()
    
    test_encoded = model.encoder_net(X_test)
    test_latent = model.fc_mu(test_encoded).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

# Configuration
n_features = latent_dim
colony_size = 30
max_cycles = 100
limit = 20
classifier = GaussianNB()

# Fitness
def fitness(sol):
    if not sol.any():
        return 0
    X_selected = train_latent[:, sol.astype(bool)]
    scores = cross_val_score(classifier, X_selected, y_train_np, cv=5, scoring='accuracy')
    return scores.mean()

# Initializing ABC structure
solutions = np.random.randint(0, 2, size=(colony_size, n_features))
fitness_vals = np.array([fitness(s) for s in solutions])
trial_counters = np.zeros(colony_size, dtype=int)

print("Started ABC")
# ABC loop
for cycle in range(max_cycles):
    print("Cycle: ", cycle)
    # Employed Bee
    for i in range(colony_size):
        k = np.random.choice([j for j in range(colony_size) if j != i])
        phi = np.random.uniform(-1, 1, size=n_features)
        candidate = solutions[i].copy()
        candidate[phi < 0] = solutions[k][phi < 0]
        fit_candidate = fitness(candidate)
        if fit_candidate > fitness_vals[i]:
            solutions[i], fitness_vals[i], trial_counters[i] = candidate, fit_candidate, 0
        else: 
            trial_counters[i] += 1
    
    # Onlooker Bee
    probs = fitness_vals / fitness_vals.sum()
    for _ in range(colony_size):
        i = np.random.choice(range(colony_size), p=probs)
        k = np.random.choice([j for j in range(colony_size) if j != i])
        phi = np.random.uniform(-1, 1, size=n_features)
        candidate = solutions[i].copy()
        candidate[phi < 0] = solutions[k][phi < 0]
        fit_candidate = fitness(candidate)
        if fit_candidate > fitness_vals[i]:
            solutions[i], fitness_vals[i], trial_counters[i] = candidate, fit_candidate, 0
        else:
            trial_counters[i] += 1
    
    # Scout Bee
    for i in range(colony_size):
        if trial_counters[i] >= limit:
            solutions[i]    = np.random.randint(0, 2, size=n_features)
            fitness_vals[i] = fitness(solutions[i])
            trial_counters[i] = 0

best_idx  = np.argmax(fitness_vals)
best_sol  = solutions[best_idx]
selected = np.where(best_sol == 1)[0]
print(f"Selected {len(selected)} features; Best CV Accuracy = {fitness_vals[best_idx]:.4f}")
print("Selected Features: ", selected)

# Running models:
X_train_selected = train_latent[:, best_sol.astype(bool)]
X_test_selected = test_latent[:, best_sol.astype(bool)]

svm_model = SVC(random_state=42)
svm_model.fit(X_train_selected, y_train_np)
svm_preds = svm_model.predict(X_test_selected)
svm_acc = accuracy_score(y_test_np, svm_preds)
print(f"SVM Accuracy on features selected by ABC: {svm_acc * 100:.4f}")

nbc_model = GaussianNB()
nbc_model.fit(X_train_selected, y_train_np)
nbc_preds = nbc_model.predict(X_test_selected)
nbc_acc = accuracy_score(y_test_np, nbc_preds)
print(f"Naïve Bayes Accuracy on features selected by ABC: {nbc_acc * 100:.4f}")

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_selected, y_train_np)
rf_preds = rf_model.predict(X_test_selected)
rf_acc = accuracy_score(y_test_np, rf_preds)
print(f"Random Forest Accuracy on features selected by ABC: {rf_acc * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.5026
Epoch [2/50], Loss: 1.2739
Epoch [3/50], Loss: 1.1730
Epoch [4/50], Loss: 1.0703
Epoch [5/50], Loss: 1.0140
Epoch [6/50], Loss: 0.9473
Epoch [7/50], Loss: 0.9152
Epoch [8/50], Loss: 0.8966
Epoch [9/50], Loss: 0.8623
Epoch [10/50], Loss: 0.8281
Epoch [11/50], Loss: 0.8784
Epoch [12/50], Loss: 0.8362
Epoch [13/50], Loss: 0.7905
Epoch [14/50], Loss: 0.8107
Epoch [15/50], Loss: 0.8291
Epoch [16/50], Loss: 0.7877
Epoch [17/50], Loss: 0.7580
Epoch [18/50], Loss: 0.7896
Epoch [19/50], Loss: 0.7030
Epoch [20/50], Loss: 0.7720
Epoch [21/50], Loss: 0.7125
Epoch [22/50], Loss: 0.7150
Epoch [23/50], Loss: 0.6951
Epoch [24/50], Loss: 0.7046
Epoch [25/50], Loss: 0.7089
Epoch [26/50], Loss: 0.6809
Epoch [27/50], Loss: 0.6957
Epoch [28/50], Loss: 0.7416
Epoch [29/50], Loss: 0.7268
Epoch [30/50], Loss: 0.6487
Epoch [31/50], Loss: 0.7653
Epoch [32/50], Loss: 0.6611
Epoch [33/50], Loss: 0.7103
Epoch [34/50], Loss: 0.7097
Epoch [35/50], Loss: 0.6936
